In [3]:
pip install -U langchain langchain-community langchain-core langchain-google-genai langchain-huggingface chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
   

In [1]:
import shutil
import os

# Delete old database folder if it exists
if os.path.exists("./chroma_db_free"):
    shutil.rmtree("./chroma_db_free")
    print("Cleared old vector database.")

In [2]:
import time

# Run your index build and queries safely
rag_service.build_vector_index()

questions = [
    "What is the main topic of this document?",
    "What are the key rules listed?"
]

for q in questions:
    result = rag_service.query(q)
    print(f"\nQuestion: {q}\nAnswer: {result.answer}")

    # Pause to avoid rate limits
    time.sleep(4)

NameError: name 'rag_service' is not defined

In [8]:
import os
import time
import logging
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("EnterpriseRAGPipeline")

@dataclass
class QueryResult:
    question: str
    answer: str
    sources: List[Dict[str, Any]]

class EnterpriseRAGService:
    def __init__(
        self,
        pdf_path: str,
        vector_db_dir: str = "./chroma_db_free",
        chunk_size: int = 1500,
        chunk_overlap: int = 200,
        k_retrievals: int = 3
    ) -> None:
        self.pdf_path = pdf_path
        self.vector_db_dir = vector_db_dir
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.k_retrievals = k_retrievals

        self._authenticate_gemini()

        logger.info("Loading Local HuggingFace Embeddings...")
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

        # UPDATED: Use gemini-2.5-flash instead of gemini-2.0-flash
        self.llm = ChatGoogleGenerativeAI(
             model="gemini-3.8-flash",
             temperature=0.0,max_retries=10
             )

        self.vector_store: Optional[Chroma] = None

    def _authenticate_gemini(self) -> None:
        try:
            api_key = userdata.get("GEMINI_API_KEY")
            if not api_key:
                raise ValueError("GEMINI_API_KEY is empty in Colab Secrets.")
            os.environ["GOOGLE_API_KEY"] = api_key
            logger.info("Gemini API key loaded successfully.")
        except Exception as e:
            logger.error("Missing GEMINI_API_KEY in Colab Secrets (🔑 sidebar icon).")
            raise RuntimeError("Configure GEMINI_API_KEY under Secrets.") from e

    def build_vector_index(self) -> None:
        if not os.path.exists(self.pdf_path):
            raise FileNotFoundError(f"PDF missing at path: '{self.pdf_path}'. Upload the file to Colab.")

        logger.info(f"Loading document: {self.pdf_path}")
        loader = PyPDFLoader(self.pdf_path)
        documents = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap
        )
        chunks = text_splitter.split_documents(documents)
        logger.info(f"Created {len(chunks)} text chunks.")

        self.vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.vector_db_dir
        )
        logger.info("Vector store created successfully.")

    def query(self, question: str) -> QueryResult:
        if self.vector_store is None:
            raise RuntimeError("Vector store not initialized. Call build_vector_index() first.")

        retriever = self.vector_store.as_retriever(search_kwargs={"k": self.k_retrievals})
        retrieved_docs = retriever.invoke(question)

        context_str = "\n\n".join(doc.page_content for doc in retrieved_docs)

        prompt = ChatPromptTemplate.from_messages([
            ("system", (
                "You are an enterprise AI assistant. Answer strictly based on the provided context. "
                "If not present in context, state 'Information not available in document.'\n\n"
                "Context:\n{context}"
            )),
            ("human", "{question}")
        ])

        rag_chain = prompt | self.llm | StrOutputParser()
        answer = rag_chain.invoke({"context": context_str, "question": question})

        sources = [
            {
                "page": doc.metadata.get("page", 0) + 1,
                "content_snippet": doc.page_content[:150].replace("\n", " ").strip() + "..."
            }
            for doc in retrieved_docs
        ]

        return QueryResult(question=question, answer=answer, sources=sources)


# EXECUTION
LOCAL_PDF_PATH = "/content/fiba-official-rules-2026-v1-1_for test.pdf"
VECTOR_STORAGE_DIR = "./chroma_db_enterprise_free"

rag_service = EnterpriseRAGService(
    pdf_path=LOCAL_PDF_PATH,
    vector_db_dir=VECTOR_STORAGE_DIR
)

rag_service.build_vector_index()

print("INTERACTIVE RAG ASSISTANT (Type 'exit' or 'quit' to stop)")
print("=" * 80 + "\n")

# 3. Interactive Input Loop
while True:
    user_query = input("\nEnter your question: ").strip()

    # Exit condition
    if user_query.lower() in ["exit", "quit", "q"]:
        print("\nExiting interactive session. Goodbye!")
        break

    if not user_query:
        print("Please enter a non-empty question.")
        continue

    try:
        # Get response from pipeline
        result = rag_service.query(user_query)

        print("\n" + "-" * 80)
        print(f"ANSWER:\n{result.answer}\n")
        print("SOURCES & CITATIONS:")
        for idx, src in enumerate(result.sources, 1):
            print(f"  [{idx}] Page {src['page']}: {src['content_snippet']}")
        print("-" * 80)

    except Exception as e:
        print(f"\nError processing query: {e}")

    # Brief pause to respect API rate limits
    time.sleep(2)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INTERACTIVE RAG ASSISTANT (Type 'exit' or 'quit' to stop)


Enter your question: disqualification a player?

--------------------------------------------------------------------------------
ANSWER:
Based on the provided context, if an excluded player (or substitute) is actively involved in a fight or any situation leading to a fight, all possible disqualification foul penalties against them shall be penalised in accordance with Article 39.3.4, sixth bullet. 

Further information regarding the general disqualification of a player is not available in the document.

SOURCES & CITATIONS:
  [1] Page 49: 40.3.5 All possible disqualification foul penalties against first assistant coach, substitute,  excluded player or an accompanying delegation member i...
  [2] Page 49: 40.3.5 All possible disqualification foul penalties against first assistant coach, substitute,  excluded player or an accompanying delegation member i...
  [3] Page 49: 40.3.5 All possible disqualification foul penalties agai

KeyboardInterrupt: 